# 1. Introducao

Esta e a **Trilha A** consolidada do experimento, dedicada a uma familia metodologica
de **proxy supervisionado em `s`**.

Objetivo tecnico central:
- treinar modelos supervisionados diretamente no rotulo observado `s`,
- comparar o ganho empirico entre `HistGradientBoosting` e `XGBoost`,
- definir um score operacional final para ranking de unlabeled,
- e incorporar interpretabilidade do melhor modelo com SHAP.

Papel editorial da Trilha A no projeto:
- servir como **referencia empirica inicial** do estudo;
- documentar o que conseguimos aprender diretamente de `P(s=1|x)`;
- produzir uma base comparavel para trilhas PU classicas e modernas;
- manter foco em ranking operacional, score e interpretabilidade.

Definicoes conceituais obrigatorias:
- `s` = rotulo observado de selecao/acesso ao credito.
- `y` = rotulo latente real de elegibilidade ou acesso estrutural, nao observado diretamente.

Consequencia metodologica:
- esta trilha aprende **algo proximo de `P(s=1|x)`**;
- ela **nao** estima diretamente `y`;
- portanto, desempenho alto em `s` nao equivale a verdade estrutural em `y`.

Estrutura interna da trilha:
- **A2** - HistGradientBoosting supervisionado em `s`;
- **A3** - XGBoost supervisionado em `s`;
- **A4** - ajuste operacional / calibracao do melhor score em `s`;
- **A5** - interpretabilidade com SHAP.

Ponte conceitual entre trilhas:
- **Trilha A** = o que aprendemos diretamente de `P(s=1|x)`;
- **Trilha B** = PU classico sob SCAR para corrigir vies de selecao;
- **Trilha C** = formulacoes PU modernas baseadas em risco.

Limitacoes que permanecem nesta trilha:
- nao observamos `y` verdadeiro;
- scores devem ser lidos como proxy supervisionado em `s`;
- SHAP interpreta drivers do padrao observado de `s`, e nao causalidade em `y`;
- calibracao em `s` melhora leitura operacional do score, mas nao resolve o problema do rotulo latente.


In [ ]:
# ---------------------------------------------------------
# Setup do experimento: reproducibilidade e utilitarios
# ---------------------------------------------------------
# Contexto metodologico:
# Esta celula centraliza o ambiente base da nova Trilha A,
# consolidando a familia supervisionada em s.
#
# Objetivo tecnico:
# - garantir reproducibilidade;
# - padronizar caminhos, seeds e configuracoes globais;
# - disponibilizar funcoes auxiliares para diagnostico,
#   avaliacao proxy, ranking e interpretabilidade;
# - preparar a execucao em modo SMOKE para iteracao rapida.
#
# Observacao metodologica importante:
# Nesta trilha, o alvo supervisionado continua sendo s.
# Portanto, os scores estimados devem ser interpretados como
# aproximacoes de P(s=1|x), e nao como estimativas causais
# diretas de P(y=1|x).

import inspect
import os
import pickle
import random
import sys
import time
import platform
from contextlib import contextmanager
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline

try:
    import shap
except ImportError as exc:
    raise ImportError(
        "Dependencia 'shap' nao encontrada. Instale com `uv add shap==0.49.1` e rode `uv sync`."
    ) from exc

try:
    import xgboost as xgb
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
    xgb_import_error = None
except Exception as exc:
    xgb = None
    XGBClassifier = None
    XGBOOST_AVAILABLE = False
    xgb_import_error = str(exc)

SEED = 42
SMOKE = True
SMOKE_MAX_ROWS = 300_000

TOP_N_RANKING = 100_000
TOP_RANK_PCTS = [0.01, 0.05, 0.10]
MIN_GROUP_ROWS = 500
MAX_SHAP_ROWS_GLOBAL = 20_000 if SMOKE else 120_000
MAX_SHAP_ROWS_TOP = 20_000 if SMOKE else 120_000
PLOT_MAX_DISPLAY = 20

np.random.seed(SEED)
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)


def resolve_repo_root(start_path: Path) -> Path:
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "dados").exists():
            return candidate
    raise FileNotFoundError("Nao foi possivel localizar a raiz do repositorio.")


@contextmanager
def timer(label: str):
    t0 = time.perf_counter()
    yield
    elapsed = time.perf_counter() - t0
    print(f"[tempo] {label}: {elapsed:.2f}s")


def memory_mb(df: pd.DataFrame) -> float:
    return float(df.memory_usage(deep=True).sum() / 1024**2)


def resumo_nulos(df: pd.DataFrame, cols: list[str], top_n: int = 15) -> pd.DataFrame:
    out = df[cols].isna().sum().sort_values(ascending=False).to_frame("n_missing")
    out["pct_missing"] = out["n_missing"] / max(len(df), 1)
    return out.head(top_n)


def get_positive_class_index(classes, positive_label=1) -> int:
    idx = np.where(np.asarray(classes) == positive_label)[0]
    if len(idx) == 0:
        raise ValueError(
            f"Classe positiva {positive_label} nao encontrada em classes_={list(classes)}"
        )
    return int(idx[0])


def recall_at_k(y: np.ndarray, score: np.ndarray, frac: float):
    k = max(1, int(np.ceil(frac * len(y))))
    order_desc = np.argsort(-score)
    top_idx = order_desc[:k]
    positives_total = int((y == 1).sum())
    positives_topk = int((y[top_idx] == 1).sum())
    recall_k = positives_topk / positives_total if positives_total > 0 else 0.0
    return k, positives_topk, recall_k


def precision_at_k(y: np.ndarray, score: np.ndarray, frac: float):
    k = max(1, int(np.ceil(frac * len(y))))
    order_desc = np.argsort(-score)
    top_idx = order_desc[:k]
    positives_topk = int((y[top_idx] == 1).sum())
    precision_k = positives_topk / k if k > 0 else 0.0
    return k, positives_topk, precision_k


def build_topk_table(y_true, score, stage_name: str) -> pd.DataFrame:
    y_np = pd.Series(y_true).astype("int8").to_numpy(copy=False)
    s_np = pd.Series(score).astype("float64").to_numpy(copy=False)

    rows = []
    for frac in [0.01, 0.05, 0.10]:
        k_r, hit_r, rec = recall_at_k(y_np, s_np, frac)
        k_p, hit_p, prec = precision_at_k(y_np, s_np, frac)
        if k_r != k_p or hit_r != hit_p:
            raise ValueError("Recall@k e Precision@k retornaram k inconsistentes.")
        rows.append(
            {
                "stage": stage_name,
                "k_frac": frac,
                "k_absoluto": k_r,
                "positivos_topk": hit_r,
                "positivos_totais": int((y_np == 1).sum()),
                "recall_at_k": rec,
                "precision_at_k": prec,
            }
        )
    return pd.DataFrame(rows)


def evaluate_proxy_scores(y_true, score) -> dict:
    y_eval = pd.Series(y_true).astype("int8").reset_index(drop=True)
    s_eval = (
        pd.to_numeric(pd.Series(score), errors="coerce")
        .fillna(0.0)
        .astype("float64")
        .reset_index(drop=True)
    )

    if len(y_eval) != len(s_eval):
        raise ValueError(
            f"Tamanho inconsistente entre y_true={len(y_eval)} e score={len(s_eval)}"
        )

    auc_s = float(roc_auc_score(y_eval, s_eval))
    topk = build_topk_table(y_eval, s_eval, stage_name="eval")

    out = {
        "auc_s": auc_s,
        "recall_at_1pct": float(topk.loc[topk["k_frac"] == 0.01, "recall_at_k"].iloc[0]),
        "recall_at_5pct": float(topk.loc[topk["k_frac"] == 0.05, "recall_at_k"].iloc[0]),
        "recall_at_10pct": float(topk.loc[topk["k_frac"] == 0.10, "recall_at_k"].iloc[0]),
        "precision_at_1pct": float(topk.loc[topk["k_frac"] == 0.01, "precision_at_k"].iloc[0]),
        "precision_at_5pct": float(topk.loc[topk["k_frac"] == 0.05, "precision_at_k"].iloc[0]),
        "precision_at_10pct": float(topk.loc[topk["k_frac"] == 0.10, "precision_at_k"].iloc[0]),
        "topk_table": topk,
    }
    return out


def score_frame_from_index(
    df_base: pd.DataFrame,
    idx,
    y_true,
    score_model,
    model_code: str,
    model_name: str,
    split_name: str,
) -> pd.DataFrame:
    out = df_base.iloc[idx][["cpf_agricultor", "id_familia", "ibge"]].copy()
    out = out.reset_index(drop=True)
    out["s"] = pd.Series(y_true).astype("int8").reset_index(drop=True)
    out["codigo_modelo"] = model_code
    out["model_name"] = model_name
    out["split"] = split_name
    out["score_model"] = pd.Series(score_model).astype("float32").reset_index(drop=True)
    return out


def orient_scores_using_validation(
    df_scores_val: pd.DataFrame,
    df_scores_test: pd.DataFrame,
    base_col: str = "score_model",
    label_col: str = "s",
):
    mean_s1_val = float(df_scores_val.loc[df_scores_val[label_col] == 1, base_col].mean())
    mean_s0_val = float(df_scores_val.loc[df_scores_val[label_col] == 0, base_col].mean())
    invert = mean_s1_val < mean_s0_val

    if invert:
        df_scores_val["score_corrigido"] = (1.0 - df_scores_val[base_col]).astype("float32")
        df_scores_test["score_corrigido"] = (1.0 - df_scores_test[base_col]).astype("float32")
    else:
        df_scores_val["score_corrigido"] = df_scores_val[base_col].astype("float32")
        df_scores_test["score_corrigido"] = df_scores_test[base_col].astype("float32")

    diag = {
        "mean_s1_val": mean_s1_val,
        "mean_s0_val": mean_s0_val,
        "score_invertido": bool(invert),
    }
    return df_scores_val, df_scores_test, diag


def sample_smoke_by_group(
    df: pd.DataFrame,
    group_col: str,
    max_rows: int,
    seed: int,
):
    if group_col not in df.columns:
        raise ValueError(f"Coluna de grupo ausente para smoke: {group_col}")

    groups = df[group_col].astype("string")
    if groups.isna().any():
        raise ValueError(f"{group_col} contem nulos; smoke por grupo nao e seguro.")

    group_sizes = (
        df.assign(_group_tmp=groups)
        .groupby("_group_tmp", observed=False)
        .size()
        .rename("n_rows")
        .reset_index()
    )

    group_sizes = group_sizes.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    group_sizes["cumsum_rows"] = group_sizes["n_rows"].cumsum()

    keep_groups = group_sizes.loc[group_sizes["cumsum_rows"] <= max_rows, "_group_tmp"].tolist()

    if not keep_groups:
        keep_groups = [str(group_sizes.iloc[0]["_group_tmp"])]
    elif len(keep_groups) < len(group_sizes):
        next_group = str(group_sizes.iloc[len(keep_groups)]["_group_tmp"])
        keep_groups.append(next_group)

    keep_groups = pd.Index(keep_groups, dtype="string")
    mask = groups.isin(keep_groups)
    df_smoke = df.loc[mask].copy().reset_index(drop=True)

    info = {
        "n_rows_original": int(len(df)),
        "n_rows_smoke": int(len(df_smoke)),
        "n_groups_smoke": int(df_smoke[group_col].nunique(dropna=False)),
        "seed": int(seed),
        "criterion": f"amostragem reprodutivel por grupos ate aproximadamente {max_rows:,} linhas",
    }
    return df_smoke, info


def make_imputed_feature_frame(
    df_base: pd.DataFrame,
    idx,
    X_array,
    feature_names: list[str],
    label_col: str = "s",
) -> pd.DataFrame:
    meta = (
        df_base.iloc[idx][["cpf_agricultor", "id_familia", "ibge", label_col]]
        .rename(columns={label_col: "s"})
        .reset_index(drop=True)
    )
    out = pd.DataFrame(X_array, columns=feature_names)
    out.insert(0, "s", meta["s"].astype("int8"))
    out.insert(0, "ibge", meta["ibge"].astype("string"))
    out.insert(0, "id_familia", meta["id_familia"].astype("string"))
    out.insert(0, "cpf_agricultor", meta["cpf_agricultor"].astype("string"))
    return out


def compute_segment_shap_xgb(
    df_segment: pd.DataFrame,
    segment_name: str,
    booster,
    feature_names: list[str],
    max_rows: int | None = None,
    seed: int = 42,
):
    if df_segment.empty:
        raise ValueError(f"Segmento vazio para SHAP: {segment_name}")

    if max_rows is not None and len(df_segment) > max_rows:
        df_used = df_segment.sample(n=max_rows, random_state=seed).copy()
        amostrado = True
    else:
        df_used = df_segment.copy()
        amostrado = False

    X_segment = df_used[feature_names].astype("float32")
    dmatrix = xgb.DMatrix(X_segment, feature_names=feature_names)

    t0 = time.perf_counter()
    contrib = booster.predict(dmatrix, pred_contribs=True)
    elapsed = time.perf_counter() - t0

    if contrib.ndim != 2 or contrib.shape[1] != len(feature_names) + 1:
        raise ValueError(
            "Formato inesperado de pred_contribs. "
            f"Esperado colunas={len(feature_names) + 1}, obtido={contrib.shape}"
        )

    shap_values = contrib[:, :-1].astype("float64", copy=False)
    base_values = contrib[:, -1].astype("float64", copy=False)

    df_importance = pd.DataFrame(
        {
            "feature": feature_names,
            "mean_abs_shap": np.abs(shap_values).mean(axis=0),
            "mean_shap": shap_values.mean(axis=0),
            "segmento": segment_name,
            "n_linhas_segmento": int(len(df_segment)),
            "n_linhas_usadas_shap": int(len(df_used)),
            "amostrado": bool(amostrado),
            "tempo_segundos": float(elapsed),
        }
    ).sort_values("mean_abs_shap", ascending=False, ignore_index=True)

    return df_used, shap_values, base_values, df_importance


ROOT = resolve_repo_root(Path.cwd().resolve())
DATA_PATH = (
    ROOT
    / "dados"
    / "dados_processados"
    / "df_caf_demografia_ufpa_pronaf_rotulo_pu.parquet"
)
OUTPUT_DIR = ROOT / "dados" / "dados_processados" / "outputs_pu"

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Arquivo ausente: {DATA_PATH}")

print("Ambiente:")
print(f"- Python: {sys.version.split()[0]}")
print(f"- Plataforma: {platform.platform()}")
print(f"- sklearn: {sklearn.__version__}")
print(f"- XGBoost disponivel: {XGBOOST_AVAILABLE}")
if not XGBOOST_AVAILABLE:
    print(f"- erro import xgboost: {xgb_import_error}")
print(f"- shap: {shap.__version__}")
print(f"- Seed global: {SEED}")
print(f"- SMOKE: {SMOKE}")
print(f"- SMOKE_MAX_ROWS: {SMOKE_MAX_ROWS:,}")
print(f"- DATA_PATH: {DATA_PATH}")
print(f"- OUTPUT_DIR: {OUTPUT_DIR}")


# 2. Carregamento dos dados

Nesta etapa lemos a base consolidada do experimento e registramos um diagnostico inicial.

Objetivos:
- validar a leitura do parquet principal;
- registrar `shape`, memoria e distribuicao de `s`;
- ativar o modo `SMOKE` de forma **reprodutivel e consistente por grupos**;
- documentar que os resultados em `SMOKE` servem para teste rapido do pipeline, e nao para conclusoes finais.


In [ ]:
# ---------------------------------------------------------
# Leitura do dataset principal e diagnostico inicial
# ---------------------------------------------------------
with timer("Leitura do parquet principal"):
    df = pd.read_parquet(DATA_PATH, engine="pyarrow")

print(f"Shape original: {df.shape}")
print(f"Colunas totais: {len(df.columns)}")
print(f"Memoria aproximada (original): {memory_mb(df):.1f} MB")

if "s" not in df.columns:
    raise ValueError("Coluna obrigatoria ausente: 's'")

n_rows_original = int(len(df))
n_groups_original = int(df["id_familia"].astype("string").nunique(dropna=False))

if SMOKE and len(df) > SMOKE_MAX_ROWS:
    df, smoke_info = sample_smoke_by_group(
        df=df,
        group_col="id_familia",
        max_rows=SMOKE_MAX_ROWS,
        seed=SEED,
    )
    print("[SMOKE] Modo de teste rapido ativado.")
    print(f"[SMOKE] Linhas originais: {smoke_info['n_rows_original']:,}")
    print(f"[SMOKE] Linhas apos smoke: {smoke_info['n_rows_smoke']:,}")
    print(f"[SMOKE] Grupos apos smoke: {smoke_info['n_groups_smoke']:,}")
    print(f"[SMOKE] Seed usada: {smoke_info['seed']}")
    print(f"[SMOKE] Criterio: {smoke_info['criterion']}")
    print(f"[SMOKE] Memoria aproximada: {memory_mb(df):.1f} MB")
else:
    smoke_info = {
        "n_rows_original": n_rows_original,
        "n_rows_smoke": int(len(df)),
        "n_groups_smoke": n_groups_original,
        "seed": int(SEED),
        "criterion": "sem amostragem; fluxo completo",
    }

print("Dtypes por coluna:")
display(df.dtypes.to_frame(name="dtype"))

s_counts = df["s"].value_counts(dropna=False).sort_index()
s_percent = (s_counts / len(df)).rename("pct")
resumo_s = pd.concat([s_counts.rename("count"), s_percent], axis=1)

print("Distribuicao de s:")
display(resumo_s)


# 3. Configuracao de modelagem

Nesta secao consolidamos:
- alvo supervisionado (`s`);
- grupo de bloqueio (`id_familia`);
- identificadores;
- colunas de metadados fora do modelo;
- e validacoes minimas de schema.

Observacao metodologica:
o alvo desta trilha e **sempre** `s`, nao o `y` latente.


In [ ]:
# ---------------------------------------------------------
# Definicao de alvo, grupos e colunas fora do modelo
# ---------------------------------------------------------
TARGET_COL = "s"
GROUP_COL = "id_familia"
ID_COLS = ["cpf_agricultor", "id_familia"]
META_COLS = ["ibge"]

required_cols = set([TARGET_COL, GROUP_COL] + ID_COLS + META_COLS)
missing_cols = sorted(required_cols - set(df.columns))
if missing_cols:
    raise ValueError(f"Colunas obrigatorias ausentes: {missing_cols}")

s_numeric = pd.to_numeric(df[TARGET_COL], errors="coerce")
if s_numeric.isna().any():
    n_nan = int(s_numeric.isna().sum())
    raise ValueError(f"Foram encontrados {n_nan} valores nao numericos em s")

invalid_mask = ~s_numeric.isin([0, 1])
if invalid_mask.any():
    invalid_values = sorted(s_numeric[invalid_mask].unique().tolist())
    raise ValueError(f"s contem valores fora de {{0,1}}: {invalid_values[:10]}")

df[TARGET_COL] = s_numeric.astype("int8")

chave_cols = [c for c in ["cpf_agricultor", "id_familia", "ibge"] if c in df.columns]
df_key_summary = pd.DataFrame(
    {
        "coluna": chave_cols,
        "n_unicos": [int(df[c].astype("string").nunique(dropna=False)) for c in chave_cols],
        "n_missing": [int(df[c].isna().sum()) for c in chave_cols],
    }
)

print("Cardinalidade e missing das chaves:")
display(df_key_summary)

print("Missing em colunas-chave:")
display(df[chave_cols + [TARGET_COL]].isna().sum().to_frame("n_missing"))


# 4. Preparacao das features

Usamos o pipeline-base da antiga Trilha C como espinha dorsal:
- criacao de flags simples de missing para latitude/longitude;
- exclusao de IDs e metadados;
- retencao apenas de features numericas;
- imputacao por mediana.

Essa base preparada sera compartilhada por:
- **A2** - HistGradientBoosting;
- **A3** - XGBoost.


In [ ]:
# ---------------------------------------------------------
# Pre-processamento comum da Trilha A
# ---------------------------------------------------------
lat_lon_flags = []
for col in ["lat_imovel_principal", "lon_imovel_principal"]:
    if col in df.columns:
        flag_col = f"st_missing_{col}"
        if flag_col not in df.columns:
            df[flag_col] = df[col].isna().astype("int8")
        lat_lon_flags.append(flag_col)

feature_cols = [
    c for c in df.columns
    if c not in ([TARGET_COL] + ID_COLS + META_COLS)
]

numeric_feature_cols = [
    c for c in feature_cols
    if pd.api.types.is_numeric_dtype(df[c])
]

non_numeric_feature_cols = [
    c for c in feature_cols
    if c not in numeric_feature_cols
]

if len(numeric_feature_cols) == 0:
    raise ValueError("Nenhuma feature numerica disponivel para modelagem")

num_cols = list(numeric_feature_cols)
imputer = SimpleImputer(strategy="median")

print(f"Features candidatas totais: {len(feature_cols)}")
print(f"Features numericas usadas: {len(numeric_feature_cols)}")
print(f"Features nao numericas excluidas: {len(non_numeric_feature_cols)}")
print("Resumo de nulos nas numericas (top 15):")
display(resumo_nulos(df, numeric_feature_cols, top_n=15))

df_feature_catalog = pd.DataFrame(
    {
        "feature": feature_cols,
        "eh_numerica": [c in numeric_feature_cols for c in feature_cols],
        "n_missing": [int(df[c].isna().sum()) for c in feature_cols],
    }
)

print("df.info() final da base antes do split:")
display(df.info())


# 5. Split por grupos

Mantemos o bloqueio por `id_familia` para evitar vazamento entre treino, validacao e teste.

Estrategia:
- `train` = 80%
- `val` = 10%
- `test` = 10%

O mesmo split sera compartilhado por A2, A3, A4 e A5.


In [ ]:
# ---------------------------------------------------------
# Split 80/10/10 com bloqueio por id_familia
# ---------------------------------------------------------
X_all = df[num_cols]
y_all = df[TARGET_COL].astype("int8")
groups_all = df[GROUP_COL].astype("string")

if groups_all.isna().any():
    n_missing_group = int(groups_all.isna().sum())
    raise ValueError(
        f"{GROUP_COL} possui {n_missing_group} nulos; split por grupos nao e seguro"
    )

gss_stage1 = GroupShuffleSplit(
    n_splits=1,
    train_size=0.8,
    test_size=0.2,
    random_state=SEED,
)
idx_train, idx_temp = next(gss_stage1.split(X_all, y=y_all, groups=groups_all))

y_temp = y_all.iloc[idx_temp]
groups_temp = groups_all.iloc[idx_temp]

gss_stage2 = GroupShuffleSplit(
    n_splits=1,
    train_size=0.5,
    test_size=0.5,
    random_state=SEED,
)
idx_val_rel, idx_test_rel = next(
    gss_stage2.split(idx_temp, y=y_temp, groups=groups_temp)
)

idx_val = idx_temp[idx_val_rel]
idx_test = idx_temp[idx_test_rel]

train_groups = pd.unique(groups_all.iloc[idx_train])
val_groups = pd.unique(groups_all.iloc[idx_val])
test_groups = pd.unique(groups_all.iloc[idx_test])

overlap_train_val = len(np.intersect1d(train_groups, val_groups))
overlap_train_test = len(np.intersect1d(train_groups, test_groups))
overlap_val_test = len(np.intersect1d(val_groups, test_groups))

assert overlap_train_val == 0, "Vazamento: familias em train e val"
assert overlap_train_test == 0, "Vazamento: familias em train e test"
assert overlap_val_test == 0, "Vazamento: familias em val e test"

X_train_raw = X_all.iloc[idx_train].copy()
X_val_raw = X_all.iloc[idx_val].copy()
X_test_raw = X_all.iloc[idx_test].copy()

y_train = y_all.iloc[idx_train].reset_index(drop=True)
y_val = y_all.iloc[idx_val].reset_index(drop=True)
y_test = y_all.iloc[idx_test].reset_index(drop=True)

def resumo_split(nome, idx_part):
    y_part = y_all.iloc[idx_part]
    groups_part = groups_all.iloc[idx_part]
    return {
        "split": nome,
        "n_linhas": int(len(idx_part)),
        "n_familias": int(groups_part.nunique(dropna=False)),
        "pct_s_1": float((y_part == 1).mean()),
        "pct_s_0": float((y_part == 0).mean()),
    }

df_split_summary = pd.DataFrame(
    [
        resumo_split("train", idx_train),
        resumo_split("val", idx_val),
        resumo_split("test", idx_test),
    ]
)

pct_s1_global = float((y_all == 1).mean())
df_split_summary["pct_s_1_global"] = pct_s1_global
df_split_summary["delta_pct_s_1_vs_global"] = (
    df_split_summary["pct_s_1"] - pct_s1_global
)

print("Resumo do split por grupos:")
display(df_split_summary)
print(
    "Overlaps por grupos:",
    f"train-val={overlap_train_val}",
    f"train-test={overlap_train_test}",
    f"val-test={overlap_val_test}",
)


# 6. Bloco A2 - HistGradientBoosting

O bloco **A2** reaproveita a espinha dorsal da antiga Trilha C e a reinsere
explicitamente como parte da nova Trilha A.

Papel metodologico do A2:
- baseline nao linear supervisionado em `s`;
- referencia inicial forte e escalavel;
- modelo puramente observavel, sem wrapper PU.


In [ ]:
# ---------------------------------------------------------
# A2 - Definicao e treinamento do HistGradientBoosting
# ---------------------------------------------------------
a2_estimator = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_depth=6,
    max_iter=600,
    random_state=SEED,
    min_samples_leaf=50,
    l2_regularization=0.1,
)

a2_model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("hgb", a2_estimator),
    ]
)

y_train_np = y_train.to_numpy(dtype="int8", copy=False)

t0 = time.perf_counter()
a2_model.fit(X_train_raw, y_train_np)
a2_train_seconds = time.perf_counter() - t0

a2_step = a2_model.named_steps["hgb"]
a2_classes = np.asarray(a2_step.classes_)
a2_pos_idx = get_positive_class_index(a2_classes, positive_label=1)
a2_n_iter = getattr(a2_step, "n_iter_", None)

X_test_imputado_a2 = a2_model.named_steps["imputer"].transform(X_test_raw)
df_X_test_imputado_a2 = make_imputed_feature_frame(
    df_base=df,
    idx=idx_test,
    X_array=X_test_imputado_a2,
    feature_names=num_cols,
    label_col=TARGET_COL,
)

print("A2 - Diagnosticos do modelo:")
print(f"- Tempo de treino: {a2_train_seconds:.2f}s")
print(f"- Classes: {a2_classes}")
print(f"- Indice da classe positiva: {a2_pos_idx}")
print(f"- n_iter_: {a2_n_iter}")


In [ ]:
# ---------------------------------------------------------
# A2 - Scores, diagnosticos, avaliacao e ranking
# ---------------------------------------------------------
a2_proba_val = np.asarray(a2_model.predict_proba(X_val_raw))
a2_proba_test = np.asarray(a2_model.predict_proba(X_test_raw))

if a2_proba_val.ndim != 2 or a2_proba_val.shape[1] <= a2_pos_idx:
    raise ValueError(
        f"A2 predict_proba invalido na validacao: shape={a2_proba_val.shape}"
    )
if a2_proba_test.ndim != 2 or a2_proba_test.shape[1] <= a2_pos_idx:
    raise ValueError(
        f"A2 predict_proba invalido no teste: shape={a2_proba_test.shape}"
    )

a2_score_val = np.nan_to_num(
    a2_proba_val[:, a2_pos_idx].astype("float64"),
    nan=0.0,
    posinf=1.0,
    neginf=0.0,
)
a2_score_test = np.nan_to_num(
    a2_proba_test[:, a2_pos_idx].astype("float64"),
    nan=0.0,
    posinf=1.0,
    neginf=0.0,
)

df_scores_a2_val = score_frame_from_index(
    df_base=df,
    idx=idx_val,
    y_true=y_val,
    score_model=a2_score_val,
    model_code="A2",
    model_name="HistGradientBoosting",
    split_name="val",
)
df_scores_a2_test = score_frame_from_index(
    df_base=df,
    idx=idx_test,
    y_true=y_test,
    score_model=a2_score_test,
    model_code="A2",
    model_name="HistGradientBoosting",
    split_name="test",
)

df_scores_a2_val, df_scores_a2_test, a2_orientation = orient_scores_using_validation(
    df_scores_val=df_scores_a2_val,
    df_scores_test=df_scores_a2_test,
    base_col="score_model",
    label_col="s",
)

print("A2 - Medias do score na validacao:")
print(f"- mean(score_model | s=1): {a2_orientation['mean_s1_val']:.6f}")
print(f"- mean(score_model | s=0): {a2_orientation['mean_s0_val']:.6f}")
print(f"- score_invertido: {a2_orientation['score_invertido']}")

print("A2 - Distribuicao de score_corrigido por s no teste:")
display(
    df_scores_a2_test.groupby("s")["score_corrigido"]
    .describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90])
)

a2_metrics_val = evaluate_proxy_scores(y_val, df_scores_a2_val["score_corrigido"])
a2_metrics_test = evaluate_proxy_scores(y_test, df_scores_a2_test["score_corrigido"])
df_a2_topk_val = a2_metrics_val["topk_table"].assign(codigo_modelo="A2")
df_a2_topk_test = a2_metrics_test["topk_table"].assign(codigo_modelo="A2")

ranking_a2_unlabeled = (
    df_scores_a2_test[df_scores_a2_test["s"] == 0]
    .sort_values("score_corrigido", ascending=False)
    .reset_index(drop=True)
)
ranking_a2_unlabeled_top100k = ranking_a2_unlabeled.head(TOP_N_RANKING).copy()

print(f"A2 - AUC_s validacao: {a2_metrics_val['auc_s']:.6f}")
print(f"A2 - AUC_s teste: {a2_metrics_test['auc_s']:.6f}")
print("A2 - Recall/Precision@k no teste:")
display(df_a2_topk_test)
print("A2 - Top 10 do ranking unlabeled:")
display(ranking_a2_unlabeled_top100k.head(10))


# 7. Bloco A3 - XGBoost

O bloco **A3** reaproveita a antiga Trilha D como segunda familia supervisionada em `s`.

Papel metodologico do A3:
- testar maior flexibilidade nao linear;
- registrar device usado, `best_iteration` e custo computacional;
- servir como candidato forte ao melhor modelo interno da Trilha A.


In [ ]:
# ---------------------------------------------------------
# A3 - Definicao do XGBoost e estrategia de early stopping
# ---------------------------------------------------------
if not XGBOOST_AVAILABLE:
    raise ImportError(
        "xgboost nao instalado. Use `uv add xgboost` e `uv sync` antes de rodar esta trilha."
    )

xgb_params = dict(
    objective="binary:logistic",
    eval_metric="auc",
    n_estimators=5000,
    learning_rate=0.1,
    max_depth=6,
    min_child_weight=1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=SEED,
    tree_method="hist",
    device="cuda",
)

fit_kwargs_common = {"eval_set": None, "verbose": False}
fit_sig = inspect.signature(XGBClassifier.fit)

if "early_stopping_rounds" in fit_sig.parameters:
    fit_kwargs_common["early_stopping_rounds"] = 50
    a3_early_mode = "fit_param"
elif "early_stopping_rounds" in XGBClassifier().get_params():
    xgb_params["early_stopping_rounds"] = 50
    a3_early_mode = "constructor_param"
else:
    fit_kwargs_common["callbacks"] = [
        xgb.callback.EarlyStopping(rounds=50, save_best=True)
    ]
    a3_early_mode = "callback"

print(f"A3 - Early stopping mode: {a3_early_mode}")


In [ ]:
# ---------------------------------------------------------
# A3 - Treinamento, diagnosticos, scores e ranking
# ---------------------------------------------------------
with timer("A3 - Imputacao train/val/test"):
    X_train_a3 = imputer.fit_transform(X_train_raw)
    X_val_a3 = imputer.transform(X_val_raw)
    X_test_a3 = imputer.transform(X_test_raw)

y_train_np = y_train.to_numpy(dtype="int8", copy=False)
y_val_np = y_val.to_numpy(dtype="int8", copy=False)
fit_kwargs = dict(fit_kwargs_common)
fit_kwargs["eval_set"] = [(X_val_a3, y_val_np)]

a3_gpu_error = None
a3_device_usado = "cuda"
try:
    a3_model = XGBClassifier(**xgb_params)
    t0 = time.perf_counter()
    a3_model.fit(X_train_a3, y_train_np, **fit_kwargs)
    a3_train_seconds = time.perf_counter() - t0
except Exception as exc:
    a3_gpu_error = str(exc)
    a3_device_usado = "cpu"
    xgb_params_cpu = dict(xgb_params)
    xgb_params_cpu["device"] = "cpu"
    a3_model = XGBClassifier(**xgb_params_cpu)
    t0 = time.perf_counter()
    a3_model.fit(X_train_a3, y_train_np, **fit_kwargs)
    a3_train_seconds = time.perf_counter() - t0

a3_classes = np.asarray(a3_model.classes_)
a3_pos_idx = get_positive_class_index(a3_classes, positive_label=1)
a3_best_iteration = getattr(a3_model, "best_iteration", None)
a3_best_score = getattr(a3_model, "best_score", None)

if hasattr(a3_model, "feature_importances_"):
    df_a3_feature_importance = pd.DataFrame(
        {
            "feature": num_cols,
            "importance": np.asarray(a3_model.feature_importances_, dtype="float64"),
        }
    ).sort_values("importance", ascending=False, ignore_index=True)
else:
    df_a3_feature_importance = pd.DataFrame(columns=["feature", "importance"])

a3_proba_val = np.asarray(a3_model.predict_proba(X_val_a3))
a3_proba_test = np.asarray(a3_model.predict_proba(X_test_a3))

if a3_proba_val.ndim != 2 or a3_proba_val.shape[1] <= a3_pos_idx:
    raise ValueError(
        f"A3 predict_proba invalido na validacao: shape={a3_proba_val.shape}"
    )
if a3_proba_test.ndim != 2 or a3_proba_test.shape[1] <= a3_pos_idx:
    raise ValueError(
        f"A3 predict_proba invalido no teste: shape={a3_proba_test.shape}"
    )

a3_score_val = np.nan_to_num(
    a3_proba_val[:, a3_pos_idx].astype("float64"),
    nan=0.0,
    posinf=1.0,
    neginf=0.0,
)
a3_score_test = np.nan_to_num(
    a3_proba_test[:, a3_pos_idx].astype("float64"),
    nan=0.0,
    posinf=1.0,
    neginf=0.0,
)

df_scores_a3_val = score_frame_from_index(
    df_base=df,
    idx=idx_val,
    y_true=y_val,
    score_model=a3_score_val,
    model_code="A3",
    model_name="XGBoost",
    split_name="val",
)
df_scores_a3_test = score_frame_from_index(
    df_base=df,
    idx=idx_test,
    y_true=y_test,
    score_model=a3_score_test,
    model_code="A3",
    model_name="XGBoost",
    split_name="test",
)

df_scores_a3_val, df_scores_a3_test, a3_orientation = orient_scores_using_validation(
    df_scores_val=df_scores_a3_val,
    df_scores_test=df_scores_a3_test,
    base_col="score_model",
    label_col="s",
)

df_X_test_imputado_a3 = make_imputed_feature_frame(
    df_base=df,
    idx=idx_test,
    X_array=X_test_a3,
    feature_names=num_cols,
    label_col=TARGET_COL,
)

a3_metrics_val = evaluate_proxy_scores(y_val, df_scores_a3_val["score_corrigido"])
a3_metrics_test = evaluate_proxy_scores(y_test, df_scores_a3_test["score_corrigido"])
df_a3_topk_val = a3_metrics_val["topk_table"].assign(codigo_modelo="A3")
df_a3_topk_test = a3_metrics_test["topk_table"].assign(codigo_modelo="A3")

ranking_a3_unlabeled = (
    df_scores_a3_test[df_scores_a3_test["s"] == 0]
    .sort_values("score_corrigido", ascending=False)
    .reset_index(drop=True)
)
ranking_a3_unlabeled_top100k = ranking_a3_unlabeled.head(TOP_N_RANKING).copy()

print("A3 - Diagnosticos do modelo:")
print(f"- Tempo de treino: {a3_train_seconds:.2f}s")
print(f"- Device usado: {a3_device_usado}")
print(f"- Classes: {a3_classes}")
print(f"- Indice da classe positiva: {a3_pos_idx}")
print(f"- best_iteration: {a3_best_iteration}")
print(f"- best_score: {a3_best_score}")
print(f"- score_invertido: {a3_orientation['score_invertido']}")
if a3_gpu_error is not None:
    print("- Fallback GPU -> CPU acionado:")
    print(a3_gpu_error[:400])

print(f"A3 - AUC_s validacao: {a3_metrics_val['auc_s']:.6f}")
print(f"A3 - AUC_s teste: {a3_metrics_test['auc_s']:.6f}")
print("A3 - Recall/Precision@k no teste:")
display(df_a3_topk_test)

print("A3 - Top 15 importancias:")
display(df_a3_feature_importance.head(15))

print("A3 - Top 10 do ranking unlabeled:")
display(ranking_a3_unlabeled_top100k.head(10))


# 8. Comparacao interna dos modelos supervisionados em `s`

Nesta secao comparamos **A2** e **A3** dentro do mesmo enquadramento metodologico:
- mesmo dataset;
- mesmo split por grupos;
- mesmo alvo supervisionado `s`;
- mesma logica de avaliacao proxy.

Criterio de escolha do vencedor interno:
1. maior `AUC_s` na validacao;
2. maior `Recall@5%` na validacao;
3. menor tempo de treino.

Esse criterio foi escolhido para equilibrar separacao em `s`, utilidade operacional
e custo computacional, sem introduzir um zoologico de modelos.


In [ ]:
# ---------------------------------------------------------
# Comparacao interna da Trilha A
# ---------------------------------------------------------
comparison_rows = [
    {
        "codigo_modelo": "A2",
        "model_name": "HistGradientBoosting",
        "familia_metodologica": "supervisionado_em_s",
        "base_estimator": "HistGradientBoostingClassifier",
        "componentes_adicionais": "imputacao_mediana",
        "auc_s_val": float(a2_metrics_val["auc_s"]),
        "auc_s_test": float(a2_metrics_test["auc_s"]),
        "recall_at_1pct_val": float(a2_metrics_val["recall_at_1pct"]),
        "recall_at_5pct_val": float(a2_metrics_val["recall_at_5pct"]),
        "recall_at_10pct_val": float(a2_metrics_val["recall_at_10pct"]),
        "precision_at_1pct_val": float(a2_metrics_val["precision_at_1pct"]),
        "precision_at_5pct_val": float(a2_metrics_val["precision_at_5pct"]),
        "precision_at_10pct_val": float(a2_metrics_val["precision_at_10pct"]),
        "recall_at_1pct_test": float(a2_metrics_test["recall_at_1pct"]),
        "recall_at_5pct_test": float(a2_metrics_test["recall_at_5pct"]),
        "recall_at_10pct_test": float(a2_metrics_test["recall_at_10pct"]),
        "precision_at_1pct_test": float(a2_metrics_test["precision_at_1pct"]),
        "precision_at_5pct_test": float(a2_metrics_test["precision_at_5pct"]),
        "precision_at_10pct_test": float(a2_metrics_test["precision_at_10pct"]),
        "train_seconds": float(a2_train_seconds),
        "converged": bool(True),
        "device_usado": "cpu",
        "best_iteration": np.nan,
        "score_invertido": bool(a2_orientation["score_invertido"]),
        "observacao_qualitativa": (
            "baseline nao linear forte e escalavel, usando score observavel em s"
        ),
    },
    {
        "codigo_modelo": "A3",
        "model_name": "XGBoost",
        "familia_metodologica": "supervisionado_em_s",
        "base_estimator": "XGBClassifier",
        "componentes_adicionais": f"imputacao_mediana + early_stopping({a3_early_mode})",
        "auc_s_val": float(a3_metrics_val["auc_s"]),
        "auc_s_test": float(a3_metrics_test["auc_s"]),
        "recall_at_1pct_val": float(a3_metrics_val["recall_at_1pct"]),
        "recall_at_5pct_val": float(a3_metrics_val["recall_at_5pct"]),
        "recall_at_10pct_val": float(a3_metrics_val["recall_at_10pct"]),
        "precision_at_1pct_val": float(a3_metrics_val["precision_at_1pct"]),
        "precision_at_5pct_val": float(a3_metrics_val["precision_at_5pct"]),
        "precision_at_10pct_val": float(a3_metrics_val["precision_at_10pct"]),
        "recall_at_1pct_test": float(a3_metrics_test["recall_at_1pct"]),
        "recall_at_5pct_test": float(a3_metrics_test["recall_at_5pct"]),
        "recall_at_10pct_test": float(a3_metrics_test["recall_at_10pct"]),
        "precision_at_1pct_test": float(a3_metrics_test["precision_at_1pct"]),
        "precision_at_5pct_test": float(a3_metrics_test["precision_at_5pct"]),
        "precision_at_10pct_test": float(a3_metrics_test["precision_at_10pct"]),
        "train_seconds": float(a3_train_seconds),
        "converged": bool(True),
        "device_usado": str(a3_device_usado),
        "best_iteration": (
            float(a3_best_iteration) if a3_best_iteration is not None else np.nan
        ),
        "score_invertido": bool(a3_orientation["score_invertido"]),
        "observacao_qualitativa": (
            "maior flexibilidade nao linear, com rastreabilidade de treino e device"
        ),
    },
]

df_comparacao_interna = pd.DataFrame(comparison_rows)
df_comparacao_interna = df_comparacao_interna.sort_values(
    ["auc_s_val", "recall_at_5pct_val", "train_seconds"],
    ascending=[False, False, True],
    ignore_index=True,
)

best_row = df_comparacao_interna.iloc[0].copy()
best_model_code = str(best_row["codigo_modelo"])
best_model_name = str(best_row["model_name"])

if best_model_code == "A2":
    best_model_obj = a2_model
    best_scores_val = df_scores_a2_val.copy()
    best_scores_test = df_scores_a2_test.copy()
    best_metrics_val = dict(a2_metrics_val)
    best_metrics_test = dict(a2_metrics_test)
    best_ranking_raw = ranking_a2_unlabeled.copy()
    best_train_seconds = float(a2_train_seconds)
    best_device = "cpu"
elif best_model_code == "A3":
    best_model_obj = a3_model
    best_scores_val = df_scores_a3_val.copy()
    best_scores_test = df_scores_a3_test.copy()
    best_metrics_val = dict(a3_metrics_val)
    best_metrics_test = dict(a3_metrics_test)
    best_ranking_raw = ranking_a3_unlabeled.copy()
    best_train_seconds = float(a3_train_seconds)
    best_device = str(a3_device_usado)
else:
    raise ValueError(f"codigo_modelo vencedor inesperado: {best_model_code}")

print("Tabela comparativa interna da Trilha A:")
display(df_comparacao_interna)
print(f"Modelo vencedor interno: {best_model_code} - {best_model_name}")
print(f"Criterio aplicado: AUC_s(val) -> Recall@5%(val) -> train_seconds")


# 9. Bloco A4 - Ajuste operacional / calibracao do melhor score

O bloco **A4** trabalha **apenas** sobre o modelo vencedor da Trilha A.

Objetivo:
- melhorar a leitura operacional do score;
- comparar score bruto orientado versus score calibrado;
- definir o **score final operacional** da trilha.

Limite interpretativo:
essa calibracao continua sendo feita em `s`, nao em `y`.
Portanto, melhora de calibracao em `s` **nao** deve ser lida como resolucao do problema do rotulo latente.


In [ ]:
# ---------------------------------------------------------
# A4 - Calibracao operacional do score vencedor
# ---------------------------------------------------------
calibrator = IsotonicRegression(out_of_bounds="clip")

winner_score_val = (
    pd.to_numeric(best_scores_val["score_corrigido"], errors="coerce")
    .fillna(0.0)
    .astype("float64")
)
winner_score_test = (
    pd.to_numeric(best_scores_test["score_corrigido"], errors="coerce")
    .fillna(0.0)
    .astype("float64")
)

y_val_np = y_val.to_numpy(dtype="int8", copy=False)
y_test_np = y_test.to_numpy(dtype="int8", copy=False)

calibrator.fit(winner_score_val, y_val_np)
score_calibrado_val = calibrator.transform(winner_score_val)
score_calibrado_test = calibrator.transform(winner_score_test)

best_scores_val = best_scores_val.copy()
best_scores_test = best_scores_test.copy()
best_scores_val["score_calibrado"] = pd.Series(score_calibrado_val).astype("float32")
best_scores_test["score_calibrado"] = pd.Series(score_calibrado_test).astype("float32")

raw_val_metrics = evaluate_proxy_scores(y_val, best_scores_val["score_corrigido"])
raw_test_metrics = evaluate_proxy_scores(y_test, best_scores_test["score_corrigido"])
cal_val_metrics = evaluate_proxy_scores(y_val, best_scores_val["score_calibrado"])
cal_test_metrics = evaluate_proxy_scores(y_test, best_scores_test["score_calibrado"])

brier_raw_val = float(brier_score_loss(y_val_np, winner_score_val))
brier_cal_val = float(brier_score_loss(y_val_np, score_calibrado_val))
brier_raw_test = float(brier_score_loss(y_test_np, winner_score_test))
brier_cal_test = float(brier_score_loss(y_test_np, score_calibrado_test))

use_calibrated_score = (
    (brier_cal_val < brier_raw_val)
    and (cal_val_metrics["auc_s"] >= raw_val_metrics["auc_s"] - 0.002)
)

score_final_operacional_col = "score_calibrado" if use_calibrated_score else "score_corrigido"
score_final_operacional_nome = (
    "score_calibrado" if use_calibrated_score else "score_corrigido"
)

best_scores_val["score_final_operacional"] = best_scores_val[
    score_final_operacional_col
].astype("float32")
best_scores_test["score_final_operacional"] = best_scores_test[
    score_final_operacional_col
].astype("float32")

df_ajuste_operacional = pd.DataFrame(
    [
        {
            "stage": "val",
            "score_variant": "score_corrigido",
            "auc_s": float(raw_val_metrics["auc_s"]),
            "brier_s": brier_raw_val,
            "recall_at_1pct": float(raw_val_metrics["recall_at_1pct"]),
            "recall_at_5pct": float(raw_val_metrics["recall_at_5pct"]),
            "recall_at_10pct": float(raw_val_metrics["recall_at_10pct"]),
            "precision_at_1pct": float(raw_val_metrics["precision_at_1pct"]),
            "precision_at_5pct": float(raw_val_metrics["precision_at_5pct"]),
            "precision_at_10pct": float(raw_val_metrics["precision_at_10pct"]),
        },
        {
            "stage": "val",
            "score_variant": "score_calibrado",
            "auc_s": float(cal_val_metrics["auc_s"]),
            "brier_s": brier_cal_val,
            "recall_at_1pct": float(cal_val_metrics["recall_at_1pct"]),
            "recall_at_5pct": float(cal_val_metrics["recall_at_5pct"]),
            "recall_at_10pct": float(cal_val_metrics["recall_at_10pct"]),
            "precision_at_1pct": float(cal_val_metrics["precision_at_1pct"]),
            "precision_at_5pct": float(cal_val_metrics["precision_at_5pct"]),
            "precision_at_10pct": float(cal_val_metrics["precision_at_10pct"]),
        },
        {
            "stage": "test",
            "score_variant": "score_corrigido",
            "auc_s": float(raw_test_metrics["auc_s"]),
            "brier_s": brier_raw_test,
            "recall_at_1pct": float(raw_test_metrics["recall_at_1pct"]),
            "recall_at_5pct": float(raw_test_metrics["recall_at_5pct"]),
            "recall_at_10pct": float(raw_test_metrics["recall_at_10pct"]),
            "precision_at_1pct": float(raw_test_metrics["precision_at_1pct"]),
            "precision_at_5pct": float(raw_test_metrics["precision_at_5pct"]),
            "precision_at_10pct": float(raw_test_metrics["precision_at_10pct"]),
        },
        {
            "stage": "test",
            "score_variant": "score_calibrado",
            "auc_s": float(cal_test_metrics["auc_s"]),
            "brier_s": brier_cal_test,
            "recall_at_1pct": float(cal_test_metrics["recall_at_1pct"]),
            "recall_at_5pct": float(cal_test_metrics["recall_at_5pct"]),
            "recall_at_10pct": float(cal_test_metrics["recall_at_10pct"]),
            "precision_at_1pct": float(cal_test_metrics["precision_at_1pct"]),
            "precision_at_5pct": float(cal_test_metrics["precision_at_5pct"]),
            "precision_at_10pct": float(cal_test_metrics["precision_at_10pct"]),
        },
    ]
)

df_scores_final_test = best_scores_test.copy()
df_scores_final_test["score_model_bruto_vencedor"] = df_scores_final_test["score_model"].astype("float32")
df_scores_final_test["score_bruto_orientado"] = df_scores_final_test["score_corrigido"].astype("float32")
df_scores_final_test["score_final_operacional_nome"] = score_final_operacional_nome

print("A4 - Comparacao entre score bruto orientado e calibrado:")
display(df_ajuste_operacional)
print(f"Score final operacional escolhido: {score_final_operacional_nome}")
print(f"Melhor modelo interno da Trilha A: {best_model_code} - {best_model_name}")


# 10. Ranking final de unlabeled da Trilha A

O ranking final usa:
- o **modelo vencedor** da comparacao interna;
- o **score final operacional** definido no bloco A4.

Esse ranking e o principal produto operacional da nova Trilha A.


In [ ]:
# ---------------------------------------------------------
# Ranking final de unlabeled da Trilha A
# ---------------------------------------------------------
ranking_final_unlabeled = (
    df_scores_final_test[df_scores_final_test["s"] == 0]
    .sort_values("score_final_operacional", ascending=False)
    .reset_index(drop=True)
)
ranking_final_unlabeled_top100k = ranking_final_unlabeled.head(TOP_N_RANKING).copy()

df_scores_final_test["ranking_global"] = (
    df_scores_final_test["score_final_operacional"]
    .rank(method="first", ascending=False)
    .astype("int64")
)

print(f"Total unlabeled no teste: {len(ranking_final_unlabeled):,}")
print(f"Top {TOP_N_RANKING:,} shape: {ranking_final_unlabeled_top100k.shape}")
print("Top 10 do ranking final:")
display(ranking_final_unlabeled_top100k.head(10))


# 11. Bloco A5 - Interpretabilidade com SHAP

Este bloco incorpora a logica da antiga interpretabilidade da Trilha D como parte interna da nova Trilha A.

Regra interpretativa obrigatoria:
- SHAP explica **drivers do score supervisionado em `s`**;
- SHAP **nao** deve ser interpretado como causalidade em `y`.

Regra operacional:
- prioridade para interpretar o modelo vencedor;
- se o vencedor nao oferecer um caminho SHAP robusto no ambiente atual,
  usamos o **XGBoost como modelo interpretavel principal da trilha** e documentamos a decisao.


In [ ]:
# ---------------------------------------------------------
# A5 - Preparacao do modelo interpretavel e SHAP global
# ---------------------------------------------------------
shap_feature_names = list(num_cols)
shap_model_code = best_model_code
shap_model_name = best_model_name
shap_engine = None
shap_fallback_reason = None

if best_model_code == "A3":
    shap_model_code = "A3"
    shap_model_name = "XGBoost"
    shap_engine = "xgboost_pred_contribs"
    booster_shap = a3_model.get_booster()
    df_base_shap = df_X_test_imputado_a3.merge(
        df_scores_final_test[["cpf_agricultor", "score_final_operacional", "s"]],
        on=["cpf_agricultor", "s"],
        how="inner",
        validate="1:1",
    )
else:
    try:
        shap_engine = "histgradientboosting_explainer"
        df_base_shap = df_X_test_imputado_a2.merge(
            df_scores_final_test[["cpf_agricultor", "score_final_operacional", "s"]],
            on=["cpf_agricultor", "s"],
            how="inner",
            validate="1:1",
        )
        _ = shap.Explainer(
            a2_model.named_steps["hgb"],
            df_base_shap[shap_feature_names].head(50).astype("float32"),
        )
    except Exception as exc:
        shap_model_code = "A3"
        shap_model_name = "XGBoost (modelo interpretavel principal da trilha)"
        shap_engine = "xgboost_pred_contribs_fallback"
        shap_fallback_reason = str(exc)
        booster_shap = a3_model.get_booster()
        df_base_shap = df_X_test_imputado_a3.merge(
            df_scores_final_test[["cpf_agricultor", "score_final_operacional", "s"]],
            on=["cpf_agricultor", "s"],
            how="inner",
            validate="1:1",
        )

if len(df_base_shap) != len(df_scores_final_test):
    raise ValueError(
        f"Merge da base SHAP alterou cardinalidade: esperado={len(df_scores_final_test)}, obtido={len(df_base_shap)}"
    )

if shap_model_code.startswith("A3"):
    df_global_used, shap_global_values, base_global_values, df_shap_global = (
        compute_segment_shap_xgb(
            df_segment=df_base_shap,
            segment_name="global_teste",
            booster=booster_shap,
            feature_names=shap_feature_names,
            max_rows=MAX_SHAP_ROWS_GLOBAL,
            seed=SEED,
        )
    )
else:
    global_source = df_base_shap.copy()
    if len(global_source) > MAX_SHAP_ROWS_GLOBAL:
        df_global_used = global_source.sample(
            n=MAX_SHAP_ROWS_GLOBAL,
            random_state=SEED,
        ).copy()
        global_amostrado = True
    else:
        df_global_used = global_source.copy()
        global_amostrado = False

    explainer_a2 = shap.Explainer(
        a2_model.named_steps["hgb"],
        df_global_used[shap_feature_names].astype("float32"),
    )
    exp_global = explainer_a2(df_global_used[shap_feature_names].astype("float32"))
    shap_global_values = np.asarray(exp_global.values, dtype="float64")
    if shap_global_values.ndim == 3:
        shap_global_values = shap_global_values[..., -1]
    base_global_values = np.asarray(exp_global.base_values, dtype="float64")
    if base_global_values.ndim > 1:
        base_global_values = base_global_values[..., -1]

    df_shap_global = pd.DataFrame(
        {
            "feature": shap_feature_names,
            "mean_abs_shap": np.abs(shap_global_values).mean(axis=0),
            "mean_shap": shap_global_values.mean(axis=0),
            "segmento": "global_teste",
            "n_linhas_segmento": int(len(global_source)),
            "n_linhas_usadas_shap": int(len(df_global_used)),
            "amostrado": bool(global_amostrado),
            "tempo_segundos": np.nan,
        }
    ).sort_values("mean_abs_shap", ascending=False, ignore_index=True)

print(f"Modelo usado no SHAP: {shap_model_code} - {shap_model_name}")
print(f"Engine SHAP: {shap_engine}")
if shap_fallback_reason is not None:
    print("Motivo do fallback interpretativo para XGBoost:")
    print(shap_fallback_reason[:500])

print("Top 20 features no SHAP global:")
display(df_shap_global.head(20))

X_global_plot = df_global_used[shap_feature_names].astype("float32")
exp_global_plot = shap.Explanation(
    values=shap_global_values,
    base_values=base_global_values,
    data=X_global_plot.to_numpy(),
    feature_names=shap_feature_names,
)

plt.figure(figsize=(11, 7))
shap.plots.beeswarm(exp_global_plot, max_display=PLOT_MAX_DISPLAY, show=False)
plt.title("Trilha A - SHAP Beeswarm Global (teste)")
plt.tight_layout()
plt.show()

plot_top = df_shap_global.head(PLOT_MAX_DISPLAY).iloc[::-1]
plt.figure(figsize=(10, 7))
plt.barh(plot_top["feature"], plot_top["mean_abs_shap"])
plt.title("Trilha A - Importancia Global (mean|SHAP|)")
plt.xlabel("mean(|SHAP|)")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


In [ ]:
# ---------------------------------------------------------
# A5 - SHAP no topo do ranking, por grupos e sintese
# ---------------------------------------------------------
df_unlabeled_shap = (
    df_base_shap.loc[df_base_shap["s"] == 0]
    .sort_values("score_final_operacional", ascending=False)
    .reset_index(drop=True)
)

if df_unlabeled_shap.empty:
    raise ValueError("Nao ha registros unlabeled para a analise SHAP da Trilha A.")

segment_tables = []
segment_cache = {}

for pct in TOP_RANK_PCTS:
    n_top = max(1, int(np.ceil(len(df_unlabeled_shap) * pct)))
    segment_name = f"top_{int(round(pct * 100))}pct_unlabeled"
    df_segment = df_unlabeled_shap.head(n_top).copy()

    if shap_model_code.startswith("A3"):
        df_used, shap_values, base_values, df_imp = compute_segment_shap_xgb(
            df_segment=df_segment,
            segment_name=segment_name,
            booster=booster_shap,
            feature_names=shap_feature_names,
            max_rows=MAX_SHAP_ROWS_TOP,
            seed=SEED,
        )
    else:
        if len(df_segment) > MAX_SHAP_ROWS_TOP:
            df_used = df_segment.sample(
                n=MAX_SHAP_ROWS_TOP,
                random_state=SEED,
            ).copy()
            seg_amostrado = True
        else:
            df_used = df_segment.copy()
            seg_amostrado = False

        explainer_seg = shap.Explainer(
            a2_model.named_steps["hgb"],
            df_used[shap_feature_names].astype("float32"),
        )
        exp_segment = explainer_seg(df_used[shap_feature_names].astype("float32"))
        shap_values = np.asarray(exp_segment.values, dtype="float64")
        if shap_values.ndim == 3:
            shap_values = shap_values[..., -1]
        base_values = np.asarray(exp_segment.base_values, dtype="float64")
        if base_values.ndim > 1:
            base_values = base_values[..., -1]

        df_imp = pd.DataFrame(
            {
                "feature": shap_feature_names,
                "mean_abs_shap": np.abs(shap_values).mean(axis=0),
                "mean_shap": shap_values.mean(axis=0),
                "segmento": segment_name,
                "n_linhas_segmento": int(len(df_segment)),
                "n_linhas_usadas_shap": int(len(df_used)),
                "amostrado": bool(seg_amostrado),
                "tempo_segundos": np.nan,
            }
        ).sort_values("mean_abs_shap", ascending=False, ignore_index=True)

    segment_tables.append(df_imp)
    segment_cache[segment_name] = {
        "df_used": df_used.reset_index(drop=True),
        "shap_values": shap_values,
        "base_values": base_values,
    }

df_shap_top = pd.concat(segment_tables, ignore_index=True)

print("Top features por segmento do ranking:")
display(
    df_shap_top.groupby("segmento", group_keys=False)
    .head(10)
    [["segmento", "feature", "mean_abs_shap", "mean_shap", "n_linhas_usadas_shap"]]
)

for segment_name in sorted(df_shap_top["segmento"].unique()):
    df_plot = (
        df_shap_top[df_shap_top["segmento"] == segment_name]
        .head(12)
        .iloc[::-1]
    )

    plt.figure(figsize=(9, 6))
    plt.barh(df_plot["feature"], df_plot["mean_abs_shap"])
    plt.title(f"Trilha A - {segment_name} (mean|SHAP|)")
    plt.xlabel("mean(|SHAP|)")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()

ref_segment_name = "top_10pct_unlabeled"
if ref_segment_name not in segment_cache:
    raise ValueError(f"Segmento de referencia ausente para grupos: {ref_segment_name}")

df_ref = segment_cache[ref_segment_name]["df_used"].reset_index(drop=True)
shap_ref = segment_cache[ref_segment_name]["shap_values"]

if len(df_ref) != shap_ref.shape[0]:
    raise ValueError("Inconsistencia entre df_ref e matriz SHAP do segmento de referencia.")

group_candidates = {
    "jovens_vs_nao_jovens": ["jovem"],
    "assentados_vs_nao_assentados": ["st_assentado_sipra"],
    "sexo": ["id_sexo"],
    "regiao": ["id_regiao"],
}

resolved_groups = {}
for group_name, candidates in group_candidates.items():
    chosen = next((c for c in candidates if c in shap_feature_names), None)
    if chosen is not None:
        resolved_groups[group_name] = chosen

group_tables = []
if resolved_groups:
    for group_name, col in resolved_groups.items():
        value_counts = df_ref[col].value_counts(dropna=False)
        for raw_value, n_rows in value_counts.items():
            if n_rows < MIN_GROUP_ROWS:
                continue

            mask = (df_ref[col] == raw_value).to_numpy()
            if mask.sum() == 0:
                continue

            shap_subset = shap_ref[mask]
            mean_abs = np.abs(shap_subset).mean(axis=0)
            mean_signed = shap_subset.mean(axis=0)

            df_group_imp = pd.DataFrame(
                {
                    "grupo": group_name,
                    "coluna_grupo": col,
                    "valor_grupo": str(raw_value),
                    "n_linhas_grupo": int(mask.sum()),
                    "feature": shap_feature_names,
                    "mean_abs_shap": mean_abs,
                    "mean_shap": mean_signed,
                }
            ).sort_values("mean_abs_shap", ascending=False, ignore_index=True)
            group_tables.append(df_group_imp)

if group_tables:
    df_shap_groups = pd.concat(group_tables, ignore_index=True)
else:
    df_shap_groups = pd.DataFrame(
        columns=[
            "grupo",
            "coluna_grupo",
            "valor_grupo",
            "n_linhas_grupo",
            "feature",
            "mean_abs_shap",
            "mean_shap",
        ]
    )

if df_shap_groups.empty:
    print("Nenhum grupo com tamanho minimo para analise SHAP por grupos.")
else:
    print("Top features por grupo:")
    display(
        df_shap_groups.groupby(["grupo", "valor_grupo"], group_keys=False)
        .head(10)
        [[
            "grupo",
            "valor_grupo",
            "n_linhas_grupo",
            "feature",
            "mean_abs_shap",
            "mean_shap",
        ]]
    )

df_global_top10 = df_shap_global.head(10).copy()
df_global_top10["fonte"] = "global"

df_top_segments_top10 = (
    df_shap_top.groupby("segmento", group_keys=False)
    .head(10)
    .copy()
)
df_top_segments_top10["fonte"] = "top_ranking"

if df_shap_groups.empty:
    df_groups_top5 = pd.DataFrame(columns=["feature", "mean_abs_shap", "fonte"])
else:
    df_groups_top5 = (
        df_shap_groups.groupby(["grupo", "valor_grupo"], group_keys=False)
        .head(5)
        .copy()
    )
    df_groups_top5["fonte"] = "grupos"

df_recurrence_source = pd.concat(
    [
        df_global_top10[["feature", "mean_abs_shap", "fonte"]],
        df_top_segments_top10[["feature", "mean_abs_shap", "fonte"]],
        df_groups_top5[["feature", "mean_abs_shap", "fonte"]],
    ],
    ignore_index=True,
)

df_shap_sintese = (
    df_recurrence_source.groupby("feature", as_index=False)
    .agg(
        aparicoes=("feature", "size"),
        fontes_distintas=("fonte", "nunique"),
        mean_abs_shap_medio=("mean_abs_shap", "mean"),
    )
    .sort_values(
        ["aparicoes", "mean_abs_shap_medio"],
        ascending=[False, False],
        ignore_index=True,
    )
)

print("Top features mais recorrentes na sintese SHAP:")
display(df_shap_sintese.head(20))


# 12. Salvamento de artefatos da Trilha A

Os artefatos finais desta trilha recebem prefixo proprio `trilhaA_...`
para evitar sobrescrita indevida de notebooks legados.

Persistimos, no minimo:
- scores de A2, A3 e do score final da trilha;
- ranking final de unlabeled;
- comparacao interna;
- metricas de ajuste operacional;
- modelo vencedor;
- artefatos para SHAP;
- manifest minimo da trilha.


In [ ]:
# ---------------------------------------------------------
# Persistencia dos artefatos da Trilha A
# ---------------------------------------------------------
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
suffix = "_smoke" if SMOKE else ""

trilhaA_a2_scores_path = OUTPUT_DIR / f"trilhaA_a2_scores_test{suffix}.parquet"
trilhaA_a3_scores_path = OUTPUT_DIR / f"trilhaA_a3_scores_test{suffix}.parquet"
trilhaA_scores_final_path = OUTPUT_DIR / f"trilhaA_scores_test{suffix}.parquet"
trilhaA_ranking_path = OUTPUT_DIR / f"trilhaA_ranking_unlabeled_top100k{suffix}.parquet"
trilhaA_compare_path = OUTPUT_DIR / f"trilhaA_comparacao_interna{suffix}.parquet"
trilhaA_score_adjust_path = OUTPUT_DIR / f"trilhaA_ajuste_operacional_score{suffix}.parquet"
trilhaA_metricas_path = OUTPUT_DIR / f"trilhaA_metricas_resumo{suffix}.parquet"
trilhaA_model_path = OUTPUT_DIR / f"trilhaA_modelo_vencedor{suffix}.pkl"
trilhaA_xgb_json_path = OUTPUT_DIR / f"trilhaA_a3_model_xgboost{suffix}.json"
trilhaA_feature_names_path = OUTPUT_DIR / f"trilhaA_feature_names{suffix}.parquet"
trilhaA_xtest_shap_path = OUTPUT_DIR / f"trilhaA_X_test_imputado_shap{suffix}.parquet"
trilhaA_shap_global_path = OUTPUT_DIR / f"trilhaA_shap_global{suffix}.parquet"
trilhaA_shap_top_path = OUTPUT_DIR / f"trilhaA_shap_top_ranking{suffix}.parquet"
trilhaA_shap_groups_path = OUTPUT_DIR / f"trilhaA_shap_grupos{suffix}.parquet"
trilhaA_shap_sintese_path = OUTPUT_DIR / f"trilhaA_shap_sintese{suffix}.parquet"
trilhaA_manifest_path = OUTPUT_DIR / f"trilhaA_artifacts_manifest{suffix}.parquet"

df_scores_a2_test.to_parquet(trilhaA_a2_scores_path, index=False)
df_scores_a3_test.to_parquet(trilhaA_a3_scores_path, index=False)
df_scores_final_test.to_parquet(trilhaA_scores_final_path, index=False)
ranking_final_unlabeled_top100k.to_parquet(trilhaA_ranking_path, index=False)
df_comparacao_interna.to_parquet(trilhaA_compare_path, index=False)
df_ajuste_operacional.to_parquet(trilhaA_score_adjust_path, index=False)

df_metricas_trilhaA = pd.DataFrame(
    [
        {
            "componente": "A2",
            "model_name": "HistGradientBoosting",
            "auc_s_val": float(a2_metrics_val["auc_s"]),
            "auc_s_test": float(a2_metrics_test["auc_s"]),
            "recall_at_1pct_test": float(a2_metrics_test["recall_at_1pct"]),
            "recall_at_5pct_test": float(a2_metrics_test["recall_at_5pct"]),
            "recall_at_10pct_test": float(a2_metrics_test["recall_at_10pct"]),
            "precision_at_1pct_test": float(a2_metrics_test["precision_at_1pct"]),
            "precision_at_5pct_test": float(a2_metrics_test["precision_at_5pct"]),
            "precision_at_10pct_test": float(a2_metrics_test["precision_at_10pct"]),
            "train_seconds": float(a2_train_seconds),
        },
        {
            "componente": "A3",
            "model_name": "XGBoost",
            "auc_s_val": float(a3_metrics_val["auc_s"]),
            "auc_s_test": float(a3_metrics_test["auc_s"]),
            "recall_at_1pct_test": float(a3_metrics_test["recall_at_1pct"]),
            "recall_at_5pct_test": float(a3_metrics_test["recall_at_5pct"]),
            "recall_at_10pct_test": float(a3_metrics_test["recall_at_10pct"]),
            "precision_at_1pct_test": float(a3_metrics_test["precision_at_1pct"]),
            "precision_at_5pct_test": float(a3_metrics_test["precision_at_5pct"]),
            "precision_at_10pct_test": float(a3_metrics_test["precision_at_10pct"]),
            "train_seconds": float(a3_train_seconds),
        },
        {
            "componente": "A4_score_final",
            "model_name": best_model_name,
            "auc_s_val": float(
                df_ajuste_operacional.loc[
                    (df_ajuste_operacional["stage"] == "val")
                    & (df_ajuste_operacional["score_variant"] == score_final_operacional_nome),
                    "auc_s",
                ].iloc[0]
            ),
            "auc_s_test": float(
                df_ajuste_operacional.loc[
                    (df_ajuste_operacional["stage"] == "test")
                    & (df_ajuste_operacional["score_variant"] == score_final_operacional_nome),
                    "auc_s",
                ].iloc[0]
            ),
            "recall_at_1pct_test": float(
                df_ajuste_operacional.loc[
                    (df_ajuste_operacional["stage"] == "test")
                    & (df_ajuste_operacional["score_variant"] == score_final_operacional_nome),
                    "recall_at_1pct",
                ].iloc[0]
            ),
            "recall_at_5pct_test": float(
                df_ajuste_operacional.loc[
                    (df_ajuste_operacional["stage"] == "test")
                    & (df_ajuste_operacional["score_variant"] == score_final_operacional_nome),
                    "recall_at_5pct",
                ].iloc[0]
            ),
            "recall_at_10pct_test": float(
                df_ajuste_operacional.loc[
                    (df_ajuste_operacional["stage"] == "test")
                    & (df_ajuste_operacional["score_variant"] == score_final_operacional_nome),
                    "recall_at_10pct",
                ].iloc[0]
            ),
            "precision_at_1pct_test": float(
                df_ajuste_operacional.loc[
                    (df_ajuste_operacional["stage"] == "test")
                    & (df_ajuste_operacional["score_variant"] == score_final_operacional_nome),
                    "precision_at_1pct",
                ].iloc[0]
            ),
            "precision_at_5pct_test": float(
                df_ajuste_operacional.loc[
                    (df_ajuste_operacional["stage"] == "test")
                    & (df_ajuste_operacional["score_variant"] == score_final_operacional_nome),
                    "precision_at_5pct",
                ].iloc[0]
            ),
            "precision_at_10pct_test": float(
                df_ajuste_operacional.loc[
                    (df_ajuste_operacional["stage"] == "test")
                    & (df_ajuste_operacional["score_variant"] == score_final_operacional_nome),
                    "precision_at_10pct",
                ].iloc[0]
            ),
            "train_seconds": float(best_train_seconds),
        },
    ]
)
df_metricas_trilhaA.to_parquet(trilhaA_metricas_path, index=False)

df_feature_names = pd.DataFrame(
    {
        "ordem_feature": np.arange(len(num_cols), dtype="int32"),
        "feature": num_cols,
    }
)
df_feature_names.to_parquet(trilhaA_feature_names_path, index=False)

df_base_shap.to_parquet(trilhaA_xtest_shap_path, index=False)
df_shap_global.to_parquet(trilhaA_shap_global_path, index=False)
df_shap_top.to_parquet(trilhaA_shap_top_path, index=False)
df_shap_groups.to_parquet(trilhaA_shap_groups_path, index=False)
df_shap_sintese.to_parquet(trilhaA_shap_sintese_path, index=False)

if hasattr(a3_model, "get_booster"):
    a3_model.get_booster().save_model(trilhaA_xgb_json_path)

model_payload = {
    "codigo_modelo_vencedor": best_model_code,
    "model_name": best_model_name,
    "score_final_operacional_nome": score_final_operacional_nome,
    "feature_names": num_cols,
    "model_obj": best_model_obj,
    "calibrator": calibrator,
}
with open(trilhaA_model_path, "wb") as f:
    pickle.dump(model_payload, f)

df_manifest = pd.DataFrame(
    [
        {
            "trilha": "A",
            "seed": int(SEED),
            "smoke": bool(SMOKE),
            "smoke_max_rows": int(SMOKE_MAX_ROWS),
            "n_rows_original": int(smoke_info["n_rows_original"]),
            "n_rows_execucao": int(smoke_info["n_rows_smoke"]),
            "n_train": int(len(idx_train)),
            "n_val": int(len(idx_val)),
            "n_test": int(len(idx_test)),
            "n_features": int(len(num_cols)),
            "modelo_vencedor": best_model_code,
            "model_name_vencedor": best_model_name,
            "score_final_operacional": score_final_operacional_nome,
            "device_vencedor": best_device,
            "modelo_shap": shap_model_code,
            "engine_shap": shap_engine,
            "scores_a2_file": trilhaA_a2_scores_path.name,
            "scores_a3_file": trilhaA_a3_scores_path.name,
            "scores_final_file": trilhaA_scores_final_path.name,
            "ranking_file": trilhaA_ranking_path.name,
            "compare_file": trilhaA_compare_path.name,
            "score_adjust_file": trilhaA_score_adjust_path.name,
            "metrics_file": trilhaA_metricas_path.name,
            "model_file": trilhaA_model_path.name,
            "feature_names_file": trilhaA_feature_names_path.name,
            "xtest_shap_file": trilhaA_xtest_shap_path.name,
            "shap_global_file": trilhaA_shap_global_path.name,
            "shap_top_file": trilhaA_shap_top_path.name,
            "shap_groups_file": trilhaA_shap_groups_path.name,
            "shap_sintese_file": trilhaA_shap_sintese_path.name,
        }
    ]
)
df_manifest.to_parquet(trilhaA_manifest_path, index=False)

print("Artefatos salvos:")
for path in [
    trilhaA_a2_scores_path,
    trilhaA_a3_scores_path,
    trilhaA_scores_final_path,
    trilhaA_ranking_path,
    trilhaA_compare_path,
    trilhaA_score_adjust_path,
    trilhaA_metricas_path,
    trilhaA_model_path,
    trilhaA_xgb_json_path,
    trilhaA_feature_names_path,
    trilhaA_xtest_shap_path,
    trilhaA_shap_global_path,
    trilhaA_shap_top_path,
    trilhaA_shap_groups_path,
    trilhaA_shap_sintese_path,
    trilhaA_manifest_path,
]:
    print(f"- {path}")


# 13. Conclusoes da Trilha A

Esta secao consolida os achados da familia supervisionada em `s`.

O que a trilha deve sintetizar:
- comportamento do HGB;
- comportamento do XGBoost;
- melhor modelo interno;
- score final operacional;
- utilidade do ranking final;
- papel do SHAP;
- limites de aprender `s` em vez de `y`.


In [ ]:
# ---------------------------------------------------------
# Conclusao executavel da Trilha A
# ---------------------------------------------------------
print("Resumo final da Trilha A")
print(f"- A2 (HistGradientBoosting) AUC_s teste: {a2_metrics_test['auc_s']:.6f}")
print(f"- A3 (XGBoost) AUC_s teste: {a3_metrics_test['auc_s']:.6f}")
print(f"- Modelo vencedor interno: {best_model_code} - {best_model_name}")
print(f"- Score final operacional: {score_final_operacional_nome}")
print(f"- Ranking final pronto: {ranking_final_unlabeled_top100k.shape}")
print(f"- Modelo usado no SHAP: {shap_model_code} - {shap_model_name}")
print("")
print("Leitura metodologica:")
print("- A Trilha A consolida uma familia supervisionada em s, nao em y.")
print("- O score final deve ser interpretado como score operacional observavel de selecao.")
print("- O ranking final serve para priorizacao analitica e operacional de unlabeled.")
print("- O bloco SHAP explica drivers do padrao observado de s; nao causalidade em y.")
print("- Esta trilha funciona como referencia empirica inicial para comparacao futura com trilhas PU classicas e modernas.")
